In [ ]:
import csv, os, shutil
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

LOG_ROOT = "/home/vscode/NullRepair/evaluation_data/logs"
APPROACHES = {
    "advanced":       "advanced-evaluation-run-gpt5.1",
    "basic":          "basic-evaluation-run-gpt5.1",
    "agent_baseline": "agent_baseline-evaluation-run-gpt5.1",
}
APPROACH_LABELS = {"advanced": "NullRepair", "basic": "SinglePrompt", "agent_baseline": "mini-SWE-agent"}

def collect_resolved_ids(log_root, approaches):
    resolved = {key: set() for key in approaches}
    for benchmark in sorted(os.listdir(log_root)):
        bp = os.path.join(log_root, benchmark)
        if not os.path.isdir(bp) or benchmark == "total":
            continue
        for key, subdir in approaches.items():
            mp = os.path.join(bp, subdir, "metrics.tsv")
            if not os.path.exists(mp):
                continue
            with open(mp, newline="", encoding="utf-8") as f:
                for row in csv.DictReader(f, delimiter="\t"):
                    if row.get("TARGET_ERROR_RESOLVED_WITHOUT_NEW_ERRORS", "false").lower() == "true":
                        resolved[key].add(f"{benchmark}/{row['ID']}")
    return resolved

resolved_no_failing_tests = collect_resolved_ids(LOG_ROOT, APPROACHES)
for key, ids in resolved_no_failing_tests.items():
    print(f"{APPROACH_LABELS[key]:20s}: {len(ids):4d} resolved")

adv, bas, abl = resolved_no_failing_tests["advanced"], resolved_no_failing_tests["basic"], resolved_no_failing_tests["agent_baseline"]

only_adv  = len(adv - bas - abl)
only_bas  = len(bas - adv - abl)
adv_bas   = len((adv & bas) - abl)
only_abl  = len(abl - adv - bas)
adv_abl   = len((adv & abl) - bas)
bas_abl   = len((bas & abl) - adv)
all_three = len(adv & bas & abl)

print(f"\nOnly NullRepair: {only_adv}, Only SinglePrompt: {only_bas}, Only mini-SWE-agent: {only_abl}")
print(f"Adv∩Bas: {adv_bas}, Adv∩Abl: {adv_abl}, Bas∩Abl: {bas_abl}, All: {all_three}")
print(f"Total unique: {len(adv | bas | abl)}")

# Overall Venn
fig, ax = plt.subplots(figsize=(8, 6))
use_tex = shutil.which("latex") is not None
plt.rcParams['text.usetex'] = use_tex
plt.rcParams["font.size"] = 11
# Use TruType fonts instead of Type 3
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

nullrepair_label = r'\textsc{NullRepair}' if use_tex else "NullRepair"

v = venn3(subsets=(only_adv, only_bas, adv_bas, only_abl, adv_abl, bas_abl, all_three),
          set_labels=(nullrepair_label, "SinglePrompt", "mini-SWE-agent"), ax=ax, alpha=0.55)
for lbl in v.set_labels:
    if lbl: lbl.set_fontsize(12); lbl.set_fontweight("bold")
for sid in ("100","010","110","001","101","011","111"):
    lbl = v.get_label_by_id(sid)
    if lbl: lbl.set_fontsize(11)
ax.set_title(f"Resolved errors", fontsize=13, pad=14)
plt.tight_layout()
plt.savefig("venn_resolved_per_patch_alt.pdf")
print("\nSaved venn_resolved_per_patch_alt.pdf")

# Per-project
benchmarks = sorted(b for b in os.listdir(LOG_ROOT) if os.path.isdir(os.path.join(LOG_ROOT, b)) and b != "total")
ncols = 4; nrows = -(-len(benchmarks) // ncols)
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 5 * nrows))
axes_flat = axes.flatten()
for idx, bm in enumerate(benchmarks):
    adv_b = {e.split("/",1)[1] for e in adv if e.startswith(bm+"/")}
    bas_b = {e.split("/",1)[1] for e in bas if e.startswith(bm+"/")}
    abl_b = {e.split("/",1)[1] for e in abl if e.startswith(bm+"/")}
    ax = axes_flat[idx]
    total_b = len(adv_b | bas_b | abl_b)
    if total_b == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_title(bm, fontsize=10, fontweight="bold"); ax.axis("off"); continue
    only_a=len(adv_b-bas_b-abl_b); only_b=len(bas_b-adv_b-abl_b); ab=len((adv_b&bas_b)-abl_b)
    only_c=len(abl_b-adv_b-bas_b); ac=len((adv_b&abl_b)-bas_b); bc=len((bas_b&abl_b)-adv_b); abc=len(adv_b&bas_b&abl_b)
    vb = venn3(subsets=(only_a,only_b,ab,only_c,ac,bc,abc), set_labels=("Adv","Basic","AgentBL"), ax=ax, alpha=0.55)
    for lbl in vb.set_labels:
        if lbl: lbl.set_fontsize(8)
    for sid in ("100","010","110","001","101","011","111"):
        lbl = vb.get_label_by_id(sid)
        if lbl: lbl.set_fontsize(9)
    ax.set_title(f"{bm}  (n={total_b})", fontsize=10, fontweight="bold")
for idx in range(len(benchmarks), len(axes_flat)):
    axes_flat[idx].axis("off")
fig.suptitle("Per-project: Errors resolved without new errors", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("venn_resolved_per_project_per_patch_alt.pdf")
print("Saved venn_resolved_per_project_per_patch_alt.pdf")


NullRepair          :  696 resolved
SinglePrompt        :  777 resolved
mini-SWE-agent      :  853 resolved

Only NullRepair: 48, Only SinglePrompt: 72, Only mini-SWE-agent: 91
Adv∩Bas: 68, Adv∩Abl: 125, Bas∩Abl: 182, All: 455
Total unique: 1041

Saved venn_resolved_per_patch_alt.pdf
Saved venn_resolved_per_project_per_patch_alt.pdf


In [ ]:
# Venn diagram Resolved Errors


import csv, os, shutil
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from venn import venn

LOG_ROOT = "/home/vscode/NullRepair/evaluation_data/logs"
APPROACHES = {
    "advanced":       "advanced-evaluation-run-gpt5.1",
    "basic":          "basic-evaluation-run-gpt5.1",
    "agent_baseline": "agent_baseline-evaluation-run-gpt5.1",
}
APPROACH_LABELS = {"advanced": "NullRepair", "basic": "SinglePrompt", "agent_baseline": "mini-SWE-agent"}

def collect_resolved_ids(log_root, approaches):
    resolved = {key: set() for key in approaches}
    for benchmark in sorted(os.listdir(log_root)):
        bp = os.path.join(log_root, benchmark)
        if not os.path.isdir(bp) or benchmark == "total":
            continue
        for key, subdir in approaches.items():
            mp = os.path.join(bp, subdir, "metrics.tsv")
            if not os.path.exists(mp):
                continue
            with open(mp, newline="", encoding="utf-8") as f:
                for row in csv.DictReader(f, delimiter="\t"):
                    if row.get("TARGET_ERROR_RESOLVED_WITHOUT_NEW_ERRORS", "false").lower() == "true":
                        resolved[key].add(f"{benchmark}/{row['ID']}")
    return resolved

resolved_no_failing_tests = collect_resolved_ids(LOG_ROOT, APPROACHES)
for key, ids in resolved_no_failing_tests.items():
    print(f"{APPROACH_LABELS[key]:20s}: {len(ids):4d} resolved")

adv, bas, abl = resolved_no_failing_tests["advanced"], resolved_no_failing_tests["basic"], resolved_no_failing_tests["agent_baseline"]

only_adv  = len(adv - bas - abl)
only_bas  = len(bas - adv - abl)
adv_bas   = len((adv & bas) - abl)
only_abl  = len(abl - adv - bas)
adv_abl   = len((adv & abl) - bas)
bas_abl   = len((bas & abl) - adv)
all_three = len(adv & bas & abl)

print(f"\nOnly NullRepair: {only_adv}, Only SinglePrompt: {only_bas}, Only mini-SWE-agent: {only_abl}")
print(f"Adv∩Bas: {adv_bas}, Adv∩Abl: {adv_abl}, Bas∩Abl: {bas_abl}, All: {all_three}")
print(f"Total unique: {len(adv | bas | abl)}")

# Overall Venn
fig, ax = plt.subplots(figsize=(8, 6))
use_tex = shutil.which("latex") is not None
plt.rcParams['text.usetex'] = use_tex
plt.rcParams["font.size"] = 11
# Use TruType fonts instead of Type 3
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

nullrepair_label = r'\textsc{NullRepair}' if use_tex else "NullRepair"

v = venn({nullrepair_label: resolved_no_failing_tests["advanced"], "SinglePrompt": resolved_no_failing_tests["basic"], "mini-SWE-agent": resolved_no_failing_tests["agent_baseline"]},
           ax=ax, alpha=0.55)
#for lbl in v.set_labels:
#    if lbl: lbl.set_fontsize(12); lbl.set_fontweight("bold")
#for sid in ("100","010","110","001","101","011","111"):
#    lbl = v.get_label_by_id(sid)
 #   if lbl: lbl.set_fontsize(11)
ax.set_title(f"Resolved errors", fontsize=13, pad=14)
plt.tight_layout()
plt.savefig("venn_resolved_per_patch.pdf")
print("\nSaved venn_resolved_per_patch.pdf")


NullRepair          :  696 resolved
SinglePrompt        :  777 resolved
mini-SWE-agent      :  853 resolved

Only NullRepair: 48, Only SinglePrompt: 72, Only mini-SWE-agent: 91
Adv∩Bas: 68, Adv∩Abl: 125, Bas∩Abl: 182, All: 455
Total unique: 1041

Saved venn_resolved_per_patch.pdf


In [ ]:
# Venn diagram resolved errors with no failing tests

import csv, os, shutil
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from venn import venn

LOG_ROOT = "/home/vscode/NullRepair/evaluation_data/logs"
APPROACHES = {
    "advanced":       "advanced-evaluation-run-gpt5.1",
    "basic":          "basic-evaluation-run-gpt5.1",
    "agent_baseline": "agent_baseline-evaluation-run-gpt5.1",
}
APPROACH_LABELS = {"advanced": "NullRepair", "basic": "SinglePrompt", "agent_baseline": "mini-SWE-agent"}

def collect_resolved_ids(log_root, approaches):
    resolved_no_failing_tests = {key: set() for key in approaches}
    for benchmark in sorted(os.listdir(log_root)):
        bp = os.path.join(log_root, benchmark)
        if not os.path.isdir(bp) or benchmark == "total":
            continue
        for key, subdir in approaches.items():
            mp = os.path.join(bp, subdir, "metrics.tsv")
            if not os.path.exists(mp):
                continue
            with open(mp, newline="", encoding="utf-8") as f:
                for row in csv.DictReader(f, delimiter="\t"):
                    if row.get("TARGET_ERROR_RESOLVED_WITHOUT_NEW_ERRORS", "false").lower() == "true" and row.get("FAILING_TESTS", "true").lower() == "false":
                        resolved_no_failing_tests[key].add(f"{benchmark}/{row['ID']}")
    return resolved_no_failing_tests

resolved_no_failing_tests = collect_resolved_ids(LOG_ROOT, APPROACHES)
for key, ids in resolved_no_failing_tests.items():
    print(f"{APPROACH_LABELS[key]:20s}: {len(ids):4d} resolved")

adv, bas, abl = resolved_no_failing_tests["advanced"], resolved_no_failing_tests["basic"], resolved_no_failing_tests["agent_baseline"]

only_adv  = len(adv - bas - abl)
only_bas  = len(bas - adv - abl)
adv_bas   = len((adv & bas) - abl)
only_abl  = len(abl - adv - bas)
adv_abl   = len((adv & abl) - bas)
bas_abl   = len((bas & abl) - adv)
all_three = len(adv & bas & abl)

print(f"\nOnly NullRepair: {only_adv}, Only SinglePrompt: {only_bas}, Only mini-SWE-agent: {only_abl}")
print(f"Adv∩Bas: {adv_bas}, Adv∩Abl: {adv_abl}, Bas∩Abl: {bas_abl}, All: {all_three}")
print(f"Total unique: {len(adv | bas | abl)}")

# Overall Venn
fig, ax = plt.subplots(figsize=(8, 6))
use_tex = shutil.which("latex") is not None
plt.rcParams['text.usetex'] = use_tex
plt.rcParams["font.size"] = 11
# Use TruType fonts instead of Type 3
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

nullrepair_label = r'\textsc{NullRepair}' if use_tex else "NullRepair"

v = venn({nullrepair_label: resolved_no_failing_tests["advanced"], "SinglePrompt": resolved_no_failing_tests["basic"], "mini-SWE-agent": resolved_no_failing_tests["agent_baseline"]},
           ax=ax, alpha=0.55)
#for lbl in v.set_labels:
#    if lbl: lbl.set_fontsize(12); lbl.set_fontweight("bold")
#for sid in ("100","010","110","001","101","011","111"):
#    lbl = v.get_label_by_id(sid)
 #   if lbl: lbl.set_fontsize(11)plt.rcParams['text.usetex'] = True
ax.set_title(f"Resolved errors without failing tests", fontsize=13, pad=14)
plt.tight_layout()
plt.savefig("venn_resolved_no_failing_tests_per_patch.pdf")
print("\nSaved venn_resolved_no_failing_tests_per_patch.pdf")



NullRepair          :  692 resolved
SinglePrompt        :  735 resolved
mini-SWE-agent      :  816 resolved

Only NullRepair: 63, Only SinglePrompt: 60, Only mini-SWE-agent: 84
Adv∩Bas: 65, Adv∩Abl: 122, Bas∩Abl: 168, All: 442
Total unique: 1004

Saved venn_resolved_no_failing_tests_per_patch.pdf


In [ ]:
# Venn diagram manually inspected errors scores 1, 2, and 3

# Following file needs to be present
input_file = "../evaluation_data/evaluation_results/manual_inspection/manual_inspection_scoring_with_classification.tsv"


import csv, os, shutil
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from venn import venn

LOG_ROOT = "/home/vscode/NullRepair/evaluation_data/logs"
APPROACHES = {
    "hash_advanced":  "advanced-evaluation-run-gpt5.1",
    "hash_basic":          "basic-evaluation-run-gpt5.1",
    "hash_agent_baseline": "agent_baseline-evaluation-run-gpt5.1",
}
APPROACH_LABELS = {"hash_advanced": "NullRepair", "hash_basic": "SinglePrompt", "hash_agent_baseline": "mini-SWE-agent"}



tools_scores_1 = {key: set() for key in APPROACHES}
tools_scores_2 = {key: set() for key in APPROACHES}
tools_scores_3 = {key: set() for key in APPROACHES}



with open(input_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        
        for row in reader:
            # Get the tool assignments for this row
            tool_a = row['Tool A']
            tool_b = row['Tool B'] 
            tool_c = row['Tool C']
            
            # Get the scores (skip if empty)
            score_a = row['SCORE (Patch A) Consolidated'].strip()
            score_b = row['SCORE (Patch B) Consolidated'].strip()
            score_c = row['SCORE (Patch C) Consolidated'].strip()
            
            # Convert to integers if valid
            scores_dict = {}
            if score_a and score_a.isdigit():
                score_a = int(score_a)
                if score_a == 1:
                    tools_scores_1[tool_a].add(row["Benchmark"] + "_" + row['ID'])
                elif score_a == 2:
                    tools_scores_2[tool_a].add(row["Benchmark"] + "_" + row['ID'])
                elif score_a == 3:
                    tools_scores_3[tool_a].add(row["Benchmark"] + "_" + row['ID'])
            else:
                print(f"Warning: Invalid score for Tool A in row ID {row['ID']}: '{score_a}'")
            
            if score_b and score_b.isdigit():
                score_b = int(score_b)
                if score_b == 1:
                    tools_scores_1[tool_b].add(row["Benchmark"] + "_" + row['ID'])
                elif score_b == 2:
                    tools_scores_2[tool_b].add(row["Benchmark"] + "_" + row['ID'])
                elif score_b == 3:
                    tools_scores_3[tool_b].add(row["Benchmark"] + "_" + row['ID'])
            else:
                print(f"Warning: Invalid score for Tool B in row ID {row['ID']}: '{score_b}'")
                
            if score_c and score_c.isdigit():
                score_c = int(score_c)
                if score_c == 1:
                    tools_scores_1[tool_c].add(row["Benchmark"] + "_" + row['ID'])
                elif score_c == 2:
                    tools_scores_2[tool_c].add(row["Benchmark"] + "_" + row['ID'])
                elif score_c == 3:
                    tools_scores_3[tool_c].add(row["Benchmark"] + "_" + row['ID'])
            else:
                print(f"Warning: Invalid score for Tool C in row ID {row['ID']}: '{score_c}'")

# Venn score 1
fig, ax = plt.subplots(figsize=(8, 6))
use_tex = shutil.which("latex") is not None
plt.rcParams['text.usetex'] = use_tex
plt.rcParams["font.size"] = 11
# Use TruType fonts instead of Type 3
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

nullrepair_label = r'\textsc{NullRepair}' if use_tex else "NullRepair"

v = venn({nullrepair_label: tools_scores_1["hash_advanced"], "SinglePrompt": tools_scores_1["hash_basic"], "mini-SWE-agent": tools_scores_1["hash_agent_baseline"]},
           ax=ax, alpha=0.55)
ax.set_title(f"Score 1 in Manual Assessment (Likely Acceptable)", fontsize=13, pad=14)
plt.tight_layout()
plt.savefig("venn_manual_inspection_score_1.pdf")
print("\nSaved venn_manual_inspection_score_1.pdf")

# Venn score 2
fig, ax = plt.subplots(figsize=(8, 6))
use_tex = shutil.which("latex") is not None
plt.rcParams['text.usetex'] = use_tex
plt.rcParams["font.size"] = 11
# Use TruType fonts instead of Type 3
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

nullrepair_label = r'\textsc{NullRepair}' if use_tex else "NullRepair"

v = venn({nullrepair_label: tools_scores_2["hash_advanced"], "SinglePrompt": tools_scores_2["hash_basic"], "mini-SWE-agent": tools_scores_2["hash_agent_baseline"]},
           ax=ax, alpha=0.55)
ax.set_title(f"Score 2 in Manual Assessment (Needs Work)", fontsize=13, pad=14)
plt.tight_layout()
plt.savefig("venn_manual_inspection_score_2.pdf")
print("\nSaved venn_manual_inspection_score_2.pdf")


# Venn score 3
fig, ax = plt.subplots(figsize=(8, 6))
use_tex = shutil.which("latex") is not None
plt.rcParams['text.usetex'] = use_tex
plt.rcParams["font.size"] = 11
# Use TruType fonts instead of Type 3
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
nullrepair_label = r'\textsc{NullRepair}' if use_tex else "NullRepair"

v = venn({nullrepair_label: tools_scores_3["hash_advanced"], "SinglePrompt": tools_scores_3["hash_basic"], "mini-SWE-agent": tools_scores_3["hash_agent_baseline"]},
           ax=ax, alpha=0.55)
ax.set_title(f"Score 3 in Manual Assessment (Likely Unacceptable)", fontsize=13, pad=14)
plt.tight_layout()
plt.savefig("venn_manual_inspection_score_3.pdf")
print("\nSaved venn_manual_inspection_score_3.pdf")


# All three in one plot
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(24, 6))
use_tex = shutil.which("latex") is not None
plt.rcParams['text.usetex'] = use_tex
plt.rcParams["font.size"] = 11
# Use TruType fonts instead of Type 3
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

nullrepair_label = r'\textsc{NullRepair}' if use_tex else "NullRepair"
v1 = venn({nullrepair_label: tools_scores_1["hash_advanced"], "SinglePrompt": tools_scores_1["hash_basic"], "mini-SWE-agent": tools_scores_1["hash_agent_baseline"]},
           ax=axes[0], alpha=0.55)
axes[0].set_title("Score 1 (Likely Acceptable)", fontsize=12, fontweight="bold")
v2 = venn({nullrepair_label: tools_scores_2["hash_advanced"], "SinglePrompt": tools_scores_2["hash_basic"], "mini-SWE-agent": tools_scores_2["hash_agent_baseline"]},
           ax=axes[1], alpha=0.55)
axes[1].set_title("Score 2 (Needs Work)", fontsize=12, fontweight="bold")
v3 = venn({nullrepair_label: tools_scores_3["hash_advanced"], "SinglePrompt": tools_scores_3["hash_basic"], "mini-SWE-agent": tools_scores_3["hash_agent_baseline"]},
           ax=axes[2], alpha=0.55)
axes[2].set_title("Score 3 (Likely Unacceptable)", fontsize=12, fontweight="bold")
plt.suptitle("Venn diagrams of manually inspected errors by score", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("venn_manual_inspection_all_scores.pdf", dpi=150, bbox_inches="tight")
print("\nSaved venn_manual_inspection_all_scores.pdf")  




Saved venn_manual_inspection_score_1.pdf

Saved venn_manual_inspection_score_2.pdf

Saved venn_manual_inspection_score_3.pdf

Saved venn_manual_inspection_all_scores.pdf
